# 汇总微博用户 ID 与 IP 属地

读取 `data/collecting/01_user_id_ip` 下的所有文件，按用户 ID 汇总属地。若同一 ID 对应多个不同属地，则排除该 ID 的全部记录；其余记录写入 `data/collecting/user_id_ip_single_region.txt`，每行格式为 `ID,属地`。

In [ ]:
from collections import defaultdict
from pathlib import Path

# 兼容从项目根目录或 project_new/src 启动 Jupyter。
data_dir = None
for base in (Path.cwd(), *Path.cwd().parents):
    for candidate in (base / "project_new/data/collecting", base / "data/collecting"):
        if (candidate / "01_user_id_ip").is_dir():
            data_dir = candidate.resolve()
            break
    if data_dir is not None:
        break
if data_dir is None:
    raise FileNotFoundError("找不到 data/collecting/01_user_id_ip；请从项目目录内启动 Jupyter。")

source_dir = data_dir / "01_user_id_ip"
output_path = data_dir / "user_id_ip_single_region.txt"
source_files = sorted(path for path in source_dir.rglob("*") if path.is_file())
if not source_files:
    raise FileNotFoundError(f"源目录没有文件：{source_dir}")

print(f"读取 {len(source_files)} 个源文件")

In [ ]:
regions_by_id = defaultdict(set)
record_count = 0

for path in source_files:
    with path.open("r", encoding="utf-8") as source:
        for line_number, raw_line in enumerate(source, start=1):
            line = raw_line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) != 2 or not parts[0] or not parts[1]:
                raise ValueError(f"记录格式错误：{path}:{line_number}")
            user_id, region = parts
            regions_by_id[user_id].add(region)
            record_count += 1

conflicting_ids = {user_id for user_id, regions in regions_by_id.items() if len(regions) > 1}
valid_pairs = sorted(
    ((user_id, next(iter(regions))) for user_id, regions in regions_by_id.items()
     if user_id not in conflicting_ids),
    key=lambda pair: pair[0],
)

print(f"读取记录：{record_count} 行")
print(f"不同用户 ID：{len(regions_by_id)} 个")
print(f"排除多属地 ID：{len(conflicting_ids)} 个")
print(f"保留单属地 ID：{len(valid_pairs)} 个")

In [ ]:
with output_path.open("w", encoding="utf-8", newline="\n") as output:
    for user_id, region in valid_pairs:
        output.write(f"{user_id},{region}\n")

print(f"已写入：{output_path}")